<a href="https://colab.research.google.com/github/Hilma-Subair/AI-Data-Analysis-MultiAgent/blob/main/SEC_Risk_Factors_Colab_Handover_CIK_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SEC EDGAR 10-K / 20-F Risk Factors — reproducible Colab handover (CIK restored)

Rebuilt from the supplied 71-cell research notebook, **including original CIK and historical predecessor discovery** (Chunks 11C, 11D, 11F, 11J, 11K), verified fiscal-year corrections (11I), successful 10-K extraction (12B, 12C), and QC review decisions (12D–12G).

**Start:** Upload your own company/ticker CSV or Excel file. The program generates the CIK mapping and historical CIK *candidate* audit. No placeholder data is required. Run a small pilot before full extraction.

**Limitations:** 20-F Item 3.D extraction has not been live validated; predecessor links and ambiguous issuer mappings need human review. All original 10-K and 20-F TXT archives are write-once with hashes and checkpointed progress.


## 1. Install dependencies (once per Colab runtime)

In [ ]:
%pip install -q edgartools pandas openpyxl requests beautifulsoup4

## 2. Configuration (edit and run once per Colab session)

Run **pilot mode** first. Change `MODE='full'` only after checking sample 10-K and 20-F sections against SEC filings. `RUN_DISCOVERY=False` skips SEC metadata discovery on restart and loads the last saved coverage CSV instead. `RETRY_ERRORS=True` retries only errors, never overwrites already-extracted TXT.


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Use a genuine identifying SEC contact; input is not written to the project files.
SEC_CONTACT_NAME = input("SEC contact name: ").strip()
SEC_CONTACT_EMAIL = input("SEC contact email: ").strip()

START_FY, END_FY = 2016, 2025
MODE = "pilot"             # "pilot" or "full"
PILOT_PER_FORM = 2
RUN_DISCOVERY = True        # False on restart: reuse stored coverage
RETRY_ERRORS = False       # True to retry only previously logged failures
REFRESH_SEC_METADATA = False  # True to refresh SEC JSON cache
DRIVE_FOLDER = "SEC_Risk_Handover"
COMPANIES_FILENAME = "company_universe.csv"


## 3. Reusable functions (run once per new Colab runtime)

These functions are self-contained. The equivalent readable Python module `sec_risk_pipeline.py` is included in the handover ZIP for source control and LLM debugging.


In [ ]:
"""SEC EDGAR annual-report Risk Factors pipeline — Colab handover.

Core approach: SEC submissions API for company/year discovery; EdgarTools for 10-K
Item 1A and filing text; conservative 20-F Item 3.D text-boundary candidate parsing.

IMPORTANT: the 20-F parser has NOT been validated on the prior 265 company-years.
Treat all 20-F results as needing review before downstream analysis.
"""

import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from datetime import date, datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# Configuration is held in the notebook's editable CONFIG cell.
# These module globals are intentionally supplied by the caller.
SUPPORTED_FORMS = ("10-K", "20-F", "40-F")

# ===========================================================
# PROJECT SETUP / SEC REQUESTS
# ===========================================================

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def project_dirs(root):
    root = Path(root)
    paths = {k: root / k for k in ("input", "cache", "coverage", "txt", "deliverables")}
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def atomic_csv(df, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".tmp", delete=False,
                                     encoding="utf-8-sig", newline="", dir=dest.parent) as f:
        tmp = Path(f.name)
        df.to_csv(f, index=False)
    os.replace(tmp, dest)


def atomic_json(obj, dest):
    dest = Path(dest)
    with tempfile.NamedTemporaryFile(mode="w", suffix=".tmp", delete=False,
                                     encoding="utf-8", dir=dest.parent) as f:
        tmp = Path(f.name)
        json.dump(obj, f, ensure_ascii=False)
    os.replace(tmp, dest)


def hash_bytes(raw):
    return hashlib.sha256(raw).hexdigest()


def safe_slug(label):
    text = unicodedata.normalize("NFKD", str(label)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text).strip("_")
    return (text or "UNKNOWN")[:70]


def normalize_ticker(value):
    return str(value).strip().upper().replace(".", "-").replace("/", "-")


def get_ciks(value):
    out = []
    for part in re.split(r"[;,|]", str(value or "")):
        part = part.strip().replace(".0", "")
        if part and re.fullmatch(r"\d{1,10}", part):
            out.append(part.zfill(10))
    return list(dict.fromkeys(out))


def split_tickers(value):
    return [normalize_ticker(t) for t in re.split(r"[;,|]", str(value or "")) if t.strip()]


def sec_identity():
    name = SEC_CONTACT_NAME.strip()
    email = SEC_CONTACT_EMAIL.strip()
    if "REPLACE" in name.upper() or "replace" in email.lower() or not re.fullmatch(
        r"[^\s@]+@[^\s@]+\.[^\s@]+", email):
        raise ValueError("Edit SEC_CONTACT_NAME and SEC_CONTACT_EMAIL before running SEC requests.")
    return f"{name} {email}"


class SecClient:
    """Sequential SEC metadata client with cache, retry and modest pacing."""
    def __init__(self, cache_dir, identity, refresh=False, min_gap=0.30):
        self.cache = Path(cache_dir)
        self.cache.mkdir(exist_ok=True, parents=True)
        self.refresh = refresh
        self.min_gap = min_gap
        self.last_request = 0
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": identity, "Accept-Encoding": "gzip, deflate",
                                     "Accept": "application/json"})

    def json(self, url, cache_name):
        target = self.cache / cache_name
        if target.exists() and not self.refresh:
            return json.loads(target.read_text(encoding="utf-8"))
        for attempt in range(5):
            delay = self.min_gap - (time.monotonic() - self.last_request)
            if delay > 0:
                time.sleep(delay)
            self.last_request = time.monotonic()
            try:
                response = self.session.get(url, timeout=50)
                if response.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"SEC returned HTTP {response.status_code}")
                response.raise_for_status()
                result = response.json()
                atomic_json(result, target)
                return result
            except Exception as exc:
                if attempt == 4:
                    raise RuntimeError(f"SEC request failed: {url}; {exc}") from exc
                time.sleep(min(2**attempt * 2, 30))


def ensure_company_template(input_path):
    path = Path(input_path)
    if not path.exists():
        example = pd.DataFrame([
            {"company_id": "DEMO_AAPL", "company_name": "Apple Inc.", "ticker": "AAPL", "cik": "", "country": "US", "historical_ciks": ""},
            {"company_id": "DEMO_MSFT", "company_name": "Microsoft Corporation", "ticker": "MSFT", "cik": "", "country": "US", "historical_ciks": ""},
            {"company_id": "DEMO_INFY", "company_name": "Infosys Limited", "ticker": "INFY", "cik": "", "country": "India", "historical_ciks": ""},
        ])
        atomic_csv(example, path)
        print("DEMO INPUT CREATED (3 companies):", path)
        print("REPLACE this CSV with your full company list before running MODE='full'.")
    return path


def load_companies(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if "ticker" not in df.columns and "tickers" in df.columns:
        df = df.rename(columns={"tickers": "ticker"})
    for col in ("company_name", "ticker", "cik", "country", "historical_ciks", "company_id"):
        if col not in df.columns:
            df[col] = ""
    if not df["company_name"].str.strip().all():
        raise ValueError("Every company requires a company_name.")
    for idx in df.index:
        if not df.at[idx, "company_id"].strip():
            df.at[idx, "company_id"] = f"C{idx+1:04d}_{safe_slug(df.at[idx,'company_name'])}"
    if not df["company_id"].is_unique:
        raise ValueError("company_id must be unique per input company.")
    return df


# Recovered from original research notebook, Chunks 11C, 11F, 11J and 11K.
# Prior candidate decisions are *review aids*, not automatic issuer-equivalence claims.
PRIOR_PROJECT_CIK_CANDIDATES = {
    "ACCENTURE PLC CLASS A": "0001467373",
    "AMERICAN TOWER REIT": "0001053507",
    "AON PLC CLASS A": "0000315293",
    "GE AEROSPACE": "0000040545",
    "GLAXOSMITHKLINE": "0001131399",
    "MARSH": "0000062709",
    "MASTERCARD CLASS A": "0001141391",
    "NIKE CLASS B": "0000320187",
    "QUALCOMM": "0000804328",
    "TEXAS INSTRUMENT": "0000097476"
}
PRIOR_PROJECT_FALSE_MATCHES = {
    "ALLIANZ": "0001034670",
    "ANGLO AMERICAN PLC": "0000006201",
    "DEUTSCHE TELEKOM N AG": "0000936340",
    "LOREAL SA": "0001627272",
    "LVMH": "0001596967",
    "SCHNEIDER ELECTRIC": "0000311337"
}
RIO_TINTO_REGISTRANTS = ("0000863064", "0000887028")
HISTORICAL_CIK_CASES = [
    dict(key="XOM", aliases=("EXXONMOBIL", "EXXON MOBIL"),
         predecessor="Exxon Mobil Corporation", cik="0000034088",
         first=2016, last=2025, relation="PRE_2026_SUCCESSOR_REGISTRANT",
         evidence="https://www.sec.gov/Archives/edgar/data/2115436/000119312526291990/d71068d8k12b.htm"),
    dict(key="AVGO", aliases=("BROADCOM",),
         predecessor="Broadcom Limited", cik="0001649338",
         first=2016, last=2017, relation="PREDECESSOR_ISSUER",
         evidence="https://www.sec.gov/Archives/edgar/data/1649338/0001649338-16-000151-index.htm"),
    dict(key="DD", aliases=("DUPONT", "DOWDUPONT"),
         predecessor="E. I. du Pont de Nemours and Company", cik="0000030554",
         first=2016, last=2016, relation="COMPLEX_MERGER_SCOPE_REVIEW",
         evidence="https://www.sec.gov/Archives/edgar/data/30554/000003055417000004/dd-12312016x10k.htm"),
    dict(key="LIN", aliases=("LINDE",),
         predecessor="Praxair Inc.", cik="0000884905",
         first=2016, last=2016, relation="ACCOUNTING_PREDECESSOR",
         evidence="https://www.sec.gov/Archives/edgar/data/884905/000088490517000007/px201610-k.htm")
]

def normalized_issuer_name(value):
    import unicodedata
    value = unicodedata.normalize("NFKD", str(value or ""))
    value = value.encode("ascii", "ignore").decode().upper()
    words = re.findall(r"[A-Z0-9]+", value)
    # Class/stock qualifiers are not issuer identity.
    while words and words[-1] in {"A", "B", "C", "REIT"} and (
        len(words)>1 and words[-2] in {"CLASS", "SHARES", "SHARE", "STOCK"}):
        words = words[:-2]
    suffixes = {"INC","INCORPORATED","CORP","CORPORATION","CO",
                "COMPANY","LTD","LIMITED","PLC","AG","NV","SE","SA"}
    while len(words)>1 and words[-1] in suffixes:
        words.pop()
    return " ".join(words)

def _historical_cases_for(company_name, tickers):
    norm = normalized_issuer_name(company_name)
    found=[]
    for case in HISTORICAL_CIK_CASES:
        # Match ticker together with the company family, or an unambiguous family name.
        name_match = any(alias in norm for alias in case["aliases"])
        if name_match and (case["key"] in tickers or len(tickers)==0 or name_match):
            found.append(case)
    return found

def resolve_companies(companies, ticker_payload):
    """SEC directory current-CIK suggestions + historically reviewed candidates.

    Does not silently convert an ambiguous ticker to a verified registrant.
    Returns one row per input company, with full candidate information for audit.
    """
    from difflib import SequenceMatcher
    records = ticker_payload.values() if isinstance(ticker_payload, dict) else ticker_payload
    by_ticker = defaultdict(set)
    by_name = defaultdict(set)
    by_cik = defaultdict(list)
    for item in records:
        tk = normalize_ticker(item.get("ticker", ""))
        title = str(item.get("title", "")).strip()
        cik = str(item["cik_str"]).zfill(10)
        if tk: by_ticker[tk].add(cik)
        by_name[normalized_issuer_name(title)].add(cik)
        by_cik[cik].append({"ticker":tk, "title":title})
    rows=[]
    for _, co in companies.iterrows():
        name=str(co["company_name"]).strip()
        name_upper=name.upper()
        cid=co["company_id"]
        tickers=set(split_tickers(co["ticker"]))
        nname=normalized_issuer_name(name)
        t_candidates=set().union(*(by_ticker.get(t,set()) for t in tickers)) if tickers else set()
        n_candidates=by_name.get(nname,set())
        proposed=sorted(t_candidates|n_candidates)
        explicit=get_ciks(co["cik"])
        original_historical=get_ciks(co["historical_ciks"])
        selected=[]
        if explicit:
            selected=explicit
            status="USER_CIK_PROVIDED_VERIFY"
        elif n_candidates and len(n_candidates)==1 and (not t_candidates or set(n_candidates)&t_candidates):
            selected=sorted(n_candidates)
            status="CIK_NAME_MATCH_PROVISIONAL"
        elif len(t_candidates)==1:
            cik=next(iter(t_candidates))
            similarities=[SequenceMatcher(None,nname,normalized_issuer_name(r["title"])).ratio()
                          for r in by_cik.get(cik,[])]
            sim=max(similarities,default=0.0)
            if name_upper in PRIOR_PROJECT_FALSE_MATCHES and cik==PRIOR_PROJECT_FALSE_MATCHES[name_upper]:
                status="REJECTED_PRIOR_TICKER_COLLISION"
            elif name_upper in PRIOR_PROJECT_CIK_CANDIDATES and cik==PRIOR_PROJECT_CIK_CANDIDATES[name_upper]:
                selected=[cik]
                status="PRIOR_PROJECT_CIK_REVIEW"
            elif sim >= 0.75:
                selected=[cik]
                status="CIK_TICKER_NAME_SIMILAR_REVIEW"
            else:
                status="CIK_TICKER_NAME_MISMATCH_REVIEW"
        elif len(t_candidates)>1:
            matched=sorted(t_candidates & n_candidates)
            if len(matched)==1:
                selected=matched
                status="CIK_TICKER_COLLISION_RESOLVED_BY_NAME"
            else:
                status="CIK_AMBIGUOUS"
        else:
            status="CIK_UNRESOLVED"
        # Prior project's special dual-registrant mapping; inspect live filings.
        if "RIO TINTO" in nname and ("RIO" in tickers or not tickers):
            selected=list(RIO_TINTO_REGISTRANTS)
            proposed=sorted(set(proposed)|set(selected))
            status="PRIOR_PROJECT_DUAL_REGISTRANT_REVIEW"
        historical=_historical_cases_for(name,tickers)
        historic_ciks=sorted(set(original_historical)|{c["cik"] for c in historical})
        labels=["%s (%s–%s; %s)"%(c["predecessor"],c["first"],c["last"],c["relation"])
                for c in historical]
        rows.append({
            "company_id":cid,"company_name":name,"ticker":co["ticker"],"country":co["country"],
            "ciks":";".join(selected),"historical_ciks":";".join(historic_ciks),
            "cik_candidates":";".join(proposed),"resolution_status":status,
            "sec_directory_names":"; ".join(sorted({r["title"] for cik in proposed for r in by_cik.get(cik,[])})),
            "prior_project_historical_candidates":"; ".join(labels),
            "historical_status":("PREDECESSOR_CANDIDATE_REVIEW" if historical else
                                 "NOT_SYSTEMATICALLY_ASSESSED")
        })
    return pd.DataFrame(rows)

def historical_cik_audit(resolved, inventory, first_fy, last_fy):
    """Historical CIK candidates found in original investigation, cross-checked against SEC inventory.

    Always labels results as CANDIDATE; never changes annual filing coverage on its own.
    """
    rows=[]
    for _, company in resolved.iterrows():
        matches=_historical_cases_for(company["company_name"], set(split_tickers(company["ticker"])))
        for case in matches:
            for fy in range(max(case["first"],first_fy), min(case["last"],last_fy)+1):
                candidates=inventory.loc[
                    inventory["cik"].eq(case["cik"])
                    & inventory["fiscal_year"].eq(str(fy))
                    & inventory["form"].isin(["10-K","20-F","40-F"])
                ]
                if candidates.empty:
                    rows.append({"company_id":company["company_id"],"company_name":company["company_name"],
                                 "fiscal_year":str(fy),"predecessor_name":case["predecessor"],
                                 "historical_cik":case["cik"],"form":"","accession_number":"",
                                 "report_date":"","status":"NO_FORM_FOUND_REVIEW",
                                 "relation":case["relation"],"evidence_url":case["evidence"]})
                for _, filing in candidates.iterrows():
                    rows.append({"company_id":company["company_id"],"company_name":company["company_name"],
                                 "fiscal_year":str(fy),"predecessor_name":case["predecessor"],
                                 "historical_cik":case["cik"],"form":filing["form"],
                                 "accession_number":filing["accession_number"],
                                 "report_date":filing["report_date"],"status":"PREDECESSOR_CANDIDATE_REVIEW",
                                 "relation":case["relation"],"evidence_url":case["evidence"]})
    cols=["company_id","company_name","fiscal_year","predecessor_name","historical_cik","form",
          "accession_number","report_date","status","relation","evidence_url"]
    return pd.DataFrame(rows,columns=cols)

def expand_parallel_arrays(data):
    fields = ("accessionNumber", "form", "filingDate", "reportDate", "primaryDocument")
    count = len(data.get("accessionNumber", []))
    result = []
    for i in range(count):
        row = {k: (data.get(k, [])[i] if i < len(data.get(k, [])) else "") for k in fields}
        result.append(row)
    return result


def discover_inventory(resolved, client, start_fy, end_fy):
    ciks = sorted({c for _, row in resolved.iterrows()
                   for col in ("ciks", "historical_ciks") for c in get_ciks(row[col])})
    filing_rows, errors = [], []
    for n, cik in enumerate(ciks, 1):
        try:
            base = client.json(f"https://data.sec.gov/submissions/CIK{cik}.json",
                               f"submissions_CIK{cik}.json")
            collection = expand_parallel_arrays(base.get("filings", {}).get("recent", {}))
            for chunk in base.get("filings", {}).get("files", []):
                if chunk.get("filingTo", "9999") < f"{start_fy}-01-01":
                    continue
                if chunk.get("filingFrom", "0000") > f"{end_fy+2}-12-31":
                    continue
                name = chunk["name"]
                older = client.json(f"https://data.sec.gov/submissions/{name}", name)
                collection += expand_parallel_arrays(older)
            for filing in collection:
                form = filing["form"]
                if form not in ("10-K", "20-F", "40-F", "10-K/A", "20-F/A", "40-F/A"):
                    continue
                accession = filing["accessionNumber"]
                if not re.fullmatch(r"\d{10}-\d{2}-\d{6}", accession):
                    continue
                fdate = filing["filingDate"]
                if fdate and not (start_fy-1 <= int(fdate[:4]) <= end_fy+2):
                    continue
                report = filing.get("reportDate", "")
                try:
                    fyear = date.fromisoformat(report).year
                    lag_days = (date.fromisoformat(fdate) - date.fromisoformat(report)).days
                    valid = 0 <= lag_days <= 550
                except (ValueError, TypeError):
                    fyear, valid = None, False
                filing_rows.append({"cik": cik, "form": form, "accession_number": accession,
                                    "filing_date": fdate, "report_date": report,
                                    "fiscal_year": str(fyear) if fyear is not None else "",
                                    "fy_metadata_valid": valid, "fy_method": "SEC_REPORT_DATE_PROVISIONAL",
                                    "primary_document": filing.get("primaryDocument", ""),
                                    "sec_url": f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession.replace('-','')}/{filing.get('primaryDocument','')}"})
        except Exception as exc:
            errors.append({"cik": cik, "error": str(exc)})
        print(f"CIK filing-history inventory: {n}/{len(ciks)}", end="\r")
    print()
    inventory = pd.DataFrame(filing_rows)
    if inventory.empty:
        inventory = pd.DataFrame(columns=["cik", "form", "accession_number", "filing_date",
                                          "report_date", "fiscal_year", "fy_metadata_valid", "fy_method", "primary_document", "sec_url"])
    else:
        inventory = inventory.drop_duplicates(subset=["cik", "accession_number"])
    return inventory, pd.DataFrame(errors, columns=["cik", "error"])


def apply_fiscal_year_corrections(inventory, path):
    """Only accession-locked, optionally expected-report-date checked corrections.

    This is essential for early January fiscal-year endings and erroneous SEC
    reportDate metadata; never infer fiscal years from filingDate alone.
    """
    columns = ["accession_number", "expected_cik", "expected_report_date",
               "corrected_fiscal_year", "note"]
    path = Path(path)
    if not path.exists():
        atomic_csv(pd.DataFrame(columns=columns), path)
    changes = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not set(columns).issubset(changes.columns):
        raise ValueError(f"FY correction file must contain {columns}")
    if changes["accession_number"].duplicated().any():
        raise ValueError("Duplicate FY correction accessions")
    out = inventory.copy()
    for _, fix in changes.iterrows():
        mask = out["accession_number"].eq(fix["accession_number"])
        if not mask.any():
            continue  # e.g. demo universe excludes Johnson & Johnson
        if mask.sum() != 1:
            raise ValueError("Correction matched several inventory rows: " + fix["accession_number"])
        index = out.index[mask][0]
        if fix["expected_cik"] and out.at[index,"cik"] != fix["expected_cik"].zfill(10):
            raise ValueError("FY correction CIK mismatch: " + fix["accession_number"])
        if fix["expected_report_date"] and out.at[index,"report_date"] != fix["expected_report_date"]:
            raise ValueError("FY correction report date mismatch: " + fix["accession_number"])
        fy = fix["corrected_fiscal_year"]
        if not re.fullmatch(r"20\d{2}", fy):
            raise ValueError("Invalid corrected fiscal year: " + str(fy))
        out.at[index,"fiscal_year"] = fy
        out.at[index,"fy_metadata_valid"] = True
        out.at[index,"fy_method"] = "REVIEWED_ACCESSION_CORRECTION"
    return out


def load_overrides(path):
    path = Path(path)
    columns = ["company_id", "fiscal_year", "cik", "accession_number", "note"]
    if not path.exists():
        atomic_csv(pd.DataFrame(columns=columns), path)
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not set(columns).issubset(df.columns):
        raise ValueError(f"Overrides file must contain: {columns}")
    if df.duplicated(["company_id", "fiscal_year"]).any():
        raise ValueError("Only one override permitted per company-year")
    bad = df[~df["fiscal_year"].str.fullmatch(r"20\d{2}")]
    if not bad.empty:
        raise ValueError("Overrides contain invalid fiscal years")
    return df


def compile_coverage(resolved, inventory, inventory_errors, overrides, first_year, last_year):
    errors = set(inventory_errors["cik"]) if not inventory_errors.empty else set()
    override_map = {(r["company_id"], str(r["fiscal_year"])): r for _, r in overrides.iterrows()}
    rows = []
    for _, co in resolved.iterrows():
        ciks = get_ciks(co["ciks"])
        historical = get_ciks(co["historical_ciks"])
        for fy in range(first_year, last_year+1):
            year = str(fy)
            row = {"company_id": co["company_id"], "company_name": co["company_name"],
                   "country": co["country"], "ticker": co["ticker"],
                   "ciks": co["ciks"], "fiscal_year": year,
                   "coverage_flag": "", "annual_form": "", "accession_number": "",
                   "cik": "", "sec_url": "", "report_date": "", "filing_date": "",
                   "source_of_match": "", "review_note": ""}
            override = override_map.get((co["company_id"], year))
            if not ciks:
                row["coverage_flag"] = co["resolution_status"]
            elif any(c in errors for c in ciks):
                row["coverage_flag"] = "INVENTORY_ERROR"
            else:
                valid_forms = inventory[(inventory["cik"].isin(ciks)) &
                                        (inventory["fiscal_year"] == year) &
                                        (inventory["form"].isin(SUPPORTED_FORMS)) &
                                        (inventory["fy_metadata_valid"] == True)]
                invalid_forms = inventory[(inventory["cik"].isin(ciks)) &
                                          (inventory["form"].isin(SUPPORTED_FORMS)) &
                                          (inventory["fiscal_year"] == year) &
                                          (inventory["fy_metadata_valid"] != True)]
                if len(valid_forms) == 1:
                    chosen = valid_forms.iloc[0]
                    row.update({"coverage_flag": "SEC_FORM_FOUND", "annual_form": chosen["form"],
                                "accession_number": chosen["accession_number"], "cik": chosen["cik"],
                                "sec_url": chosen["sec_url"], "report_date": chosen["report_date"],
                                "filing_date": chosen["filing_date"], "source_of_match": "SEC_REPORT_DATE"})
                elif len(valid_forms) > 1:
                    row["coverage_flag"] = "AMBIGUOUS_MULTIPLE_FILINGS"
                    row["review_note"] = "; ".join(valid_forms["accession_number"].tolist())
                elif len(invalid_forms) > 0:
                    row["coverage_flag"] = "FY_ASSIGNMENT_REVIEW"
                    row["review_note"] = "SEC reportDate/filingDate difference anomalous"
                else:
                    preceding = inventory[(inventory["cik"].isin(historical)) &
                                          (inventory["fiscal_year"] == year) &
                                          (inventory["form"].isin(SUPPORTED_FORMS)) &
                                          (inventory["fy_metadata_valid"] == True)]
                    if len(preceding) == 1:
                        row["coverage_flag"] = "PREDECESSOR_CANDIDATE"
                        row["review_note"] = "Needs historical continuity review: " + preceding.iloc[0]["accession_number"]
                    elif len(preceding) > 1:
                        row["coverage_flag"] = "PREDECESSOR_AMBIGUOUS"
                        row["review_note"] = "; ".join(preceding["accession_number"].tolist())
                    else:
                        row["coverage_flag"] = "NO_FORM_IN_MATCHED_CIK"
            if override is not None:
                matching = inventory[(inventory["accession_number"] == override["accession_number"]) &
                                     (inventory["cik"] == str(override["cik"]).zfill(10)) &
                                     (inventory["form"].isin(SUPPORTED_FORMS))]
                if len(matching) != 1:
                    raise ValueError(f"Override not verified in SEC inventory: {co['company_id']} FY{year}")
                if str(override["cik"]).zfill(10) not in set(ciks + historical):
                    raise ValueError("Override CIK not in current or historical CIKs")
                chosen = matching.iloc[0]
                row.update({"coverage_flag": "SEC_FORM_FOUND", "annual_form": chosen["form"],
                            "accession_number": chosen["accession_number"], "cik": chosen["cik"],
                            "sec_url": chosen["sec_url"], "report_date": chosen["report_date"],
                            "filing_date": chosen["filing_date"], "source_of_match": "MANUAL_OVERRIDE_VERIFIED",
                            "review_note": override["note"]})
            rows.append(row)
    out = pd.DataFrame(rows)
    assert not out.duplicated(["company_id", "fiscal_year"]).any()
    return out


def read_known_issues(path):
    columns = ["accession_number", "processing_decision", "reason"]
    path = Path(path)
    if not path.exists():
        atomic_csv(pd.DataFrame(columns=columns), path)
    issues = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not set(columns).issubset(issues.columns):
        raise ValueError(f"Known issues CSV must include {columns}")
    if issues["accession_number"].duplicated().any():
        raise ValueError("Duplicate known issue accession")
    if not issues["processing_decision"].isin(["EXCLUDE_ERROR", "HOLD_FOR_REVIEW", "INCLUDE_WITH_FLAG"]).all():
        raise ValueError("Invalid known issue processing decision")
    return {r["accession_number"]: (r["processing_decision"],r["reason"]) for _,r in issues.iterrows()}


# ===========================================================
# RISK-SECTION EXTRACTION / QC
# ===========================================================

# Edgartools .obj().risk_factors is proven from the earlier 10-K pilot.
# 20-F risk section logic below is NEW and requires a per-company pilot.
TENK_START = re.compile(r"(?im)^\s*(?:item\s*1\s*a\s*[.:\-]?\s*)?risk\s+factors\s*$")
TENK_END = re.compile(r"(?im)^\s*(?:item\s*1\s*b\b[^\n]*|item\s*2\b[^\n]*|legal\s+proceedings\s*)$")
TWENTYF_START = re.compile(
    r"(?im)^\s*(?:item\s*3\s*[.\-:]?\s*)?(?:3\s*[.\-:]?\s*)?d\s*[.\-:]?\s*risk\s+factors\s*$|^\s*risk\s+factors\s*$"
)
TWENTYF_END = re.compile(
    r"(?im)^\s*(?:item\s*4\s*[.\-:]?\s*(?:information\s+(?:on|about)\s+the\s+company)?"
    r"|4\s*[.\-:]?\s*information\s+(?:on|about)\s+the\s+company)\s*$"
)
NEXT_10K_SECTION = re.compile(r"(?im)^\s*(?:item\s*1\s*b\b[^\n]*|item\s*2\b[^\n]*|signatures\s*)$")


def bounded_extract(text, start_pattern, end_pattern, min_words, max_words):
    """Avoid table-of-contents short matches; refuse ambiguous full-document matches."""
    starts = list(start_pattern.finditer(text))
    ends = list(end_pattern.finditer(text))
    candidates = []
    for start in starts:
        for end in ends:
            if end.start() <= start.end():
                continue
            chunk = text[start.start():end.start()].strip()
            words = len(chunk.split())
            if min_words <= words <= max_words:
                candidates.append((chunk, end.group().strip(), words))
            break  # only the immediately following end marker is considered
    # Duplicate identical sections should count as only one candidate.
    unique = {hash_bytes(c[0].encode("utf-8")): c for c in candidates}
    if len(unique) != 1:
        raise ValueError(f"Could not isolate one unique bounded Risk Factors section (plausible candidates: {len(unique)}).")
    return next(iter(unique.values()))


def extract_risk_section(filing, form):
    """Return text, exact method, QC seed. Never archive an unbounded full filing.

    10-K structured extraction was used successfully for 486 filings in source
    notebook Chunk 12B (not all passed accuracy checks). The 20-F methods
    remain UNVALIDATED on the earlier 265 identified form-year records.
    """
    if form not in ("10-K", "20-F"):
        raise ValueError(f"No section extractor implemented for {form}")

    structured_error = ""
    try:
        obj = filing.obj()
        section = getattr(obj, "risk_factors", None)
        text = str(section).strip() if section is not None else ""
        if len(text.split()) >= 300:
            if form == "10-K":
                return text, "EDGARTOOLS_TENK_RISK_FACTORS", "QC_UNCHECKED"
            return text, "EDGARTOOLS_TWENTYF_RISK_FACTORS_CANDIDATE", "REVIEW_20F_UNVALIDATED"
    except Exception as error:
        structured_error = f"{type(error).__name__}: {error}"

    start_pattern = TENK_START if form == "10-K" else TWENTYF_START
    end_pattern = TENK_END if form == "10-K" else TWENTYF_END
    min_words = 800
    max_words = 35000 if form == "10-K" else 60000

    errors = []
    try:
        full_text = filing.text() or ""
        candidate, _, _ = bounded_extract(str(full_text), start_pattern, end_pattern, min_words, max_words)
        base = "REVIEW_FALLBACK" if form == "10-K" else "REVIEW_20F_UNVALIDATED"
        return candidate, "EDGARTOOLS_FULL_TEXT_BOUNDED_FALLBACK", base
    except Exception as exc:
        errors.append(str(exc))

    # Exact McDonald's 2020–2022 recovery approach from original Chunk 12C.
    try:
        from bs4 import BeautifulSoup
        html = filing.html() or ""
        soup = BeautifulSoup(html, "html.parser")
        for element in soup(["script", "style", "ix:header"]):
            element.decompose()
        full_text = soup.get_text(separator="\n", strip=True)
        candidate, _, _ = bounded_extract(full_text, start_pattern, end_pattern, min_words, max_words)
        base = "REVIEW_FALLBACK" if form == "10-K" else "REVIEW_20F_UNVALIDATED"
        return candidate, "EDGARTOOLS_HTML_BOUNDED_FALLBACK", base
    except Exception as exc:
        errors.append(str(exc))
    raise ValueError("Could not identify one bounded Risk Factors section. "
                     + " | ".join(errors)[:500] + (" | primary="+structured_error[:120] if structured_error else ""))


def quality_check(text, form, base_qc):
    words = len(text.split())
    reasons = []
    if words < 1000:
        reasons.append("VERY_SHORT")
    if words > 30000:
        reasons.append("VERY_LONG")
    if form == "10-K":
        if re.search(r"\brisk\s+factors\b", text[:900], re.I) is None:
            reasons.append("REVIEW_START")
        if NEXT_10K_SECTION.search(text):
            reasons.append("REVIEW_END")
    elif form == "20-F":
        if re.search(r"\brisk\s+factors\b", text[:500], re.I) is None:
            reasons.append("REVIEW_START")
        if TWENTYF_END.search(text):
            reasons.append("REVIEW_END")
    if base_qc != "QC_UNCHECKED":
        reasons.insert(0, base_qc)
    status = "PASSED_BASIC_CHECKS" if not reasons else ";".join(dict.fromkeys(reasons))
    return status, words


def expected_txt_path(txt_dir, record):
    slug = safe_slug(record["company_name"]) + "__" + safe_slug(record["company_id"])
    form = record["annual_form"]
    return Path(txt_dir) / form / slug / f"FY{record['fiscal_year']}" / (record["accession_number"].replace("-", "") + "_RiskFactors.txt")


def process_extractions(coverage, txt_dir, checkpoint_path, mode="pilot", per_form=2, retry_errors=False):
    """Uses SEC filing accession IDs; atomically checkpoints each attempt."""
    from edgar import get_by_accession_number, set_identity
    set_identity(sec_identity())
    checkpoint_path = Path(checkpoint_path)
    columns = ("company_id", "company_name", "fiscal_year", "annual_form", "cik",
               "accession_number", "sec_url", "extraction_status", "qc_status",
               "extraction_method", "word_count", "sha256", "txt_path", "error_message", "processed_utc")
    if checkpoint_path.exists():
        progress = pd.read_csv(checkpoint_path, dtype=str, keep_default_na=False)
    else:
        progress = pd.DataFrame(columns=columns)
    # Compound key prevents distinct company rows sharing one accession from colliding.
    def progress_key(record):
        return (str(record["company_id"]), str(record["fiscal_year"]),
                str(record["accession_number"]))

    existing = {progress_key(r): r for _, r in progress.iterrows()}
    if len(existing) != len(progress):
        raise ValueError("Duplicate company/year/accession rows in checkpoint")
    queue = coverage[(coverage["coverage_flag"] == "SEC_FORM_FOUND") &
                     (coverage["annual_form"].isin(["10-K", "20-F"]))].copy()
    queue = queue.sort_values(["fiscal_year", "company_name"], ascending=[False, True])
    done = Counter()
    for _, r in queue.iterrows():
        accession = r["accession_number"]
        key = progress_key(r)
        old = existing.get(key)
        if old is not None:
            if old["extraction_status"] == "EXTRACTED":
                if str(old["cik"]).zfill(10) != str(r["cik"]).zfill(10) or old["annual_form"] != r["annual_form"]:
                    raise RuntimeError(f"Checkpoint does not match current filing metadata: {key}")
                p = Path(old["txt_path"])
                if not p.is_file() or hash_bytes(p.read_bytes()) != old["sha256"]:
                    raise RuntimeError(f"Archived text integrity failure: {p}")
                continue
            if not retry_errors:
                continue
        form = r["annual_form"]
        if mode == "pilot" and done[form] >= per_form:
            continue
        done[form] += 1
        result = {col: str(r.get(col, "")) for col in columns}
        result.update({"extraction_status": "ERROR", "qc_status": "", "word_count": "",
                       "sha256": "", "txt_path": "", "error_message": "",
                       "processed_utc": now_utc(), "extraction_method": ""})
        try:
            filing = get_by_accession_number(accession)
            if filing is None:
                raise ValueError("Accession not found")
            if str(filing.cik).zfill(10) != str(r["cik"]).zfill(10):
                raise ValueError("Filing CIK does not match inventory")
            if filing.form != form:
                raise ValueError("Filing form does not match inventory")
            returned_accession = getattr(filing, "accession_no", None)
            if returned_accession is not None and str(returned_accession) != accession:
                raise ValueError("Returned filing accession does not match inventory")
            text, method, base_qc = extract_risk_section(filing, form)
            qc, words = quality_check(text, form, base_qc)
            if words < 300:
                raise ValueError("Extracted fewer than 300 words")
            output_path = expected_txt_path(txt_dir, r)
            output_path.parent.mkdir(exist_ok=True, parents=True)
            raw = text.encode("utf-8")
            if output_path.exists():
                if hash_bytes(output_path.read_bytes()) != hash_bytes(raw):
                    raise FileExistsError("Different existing text file: do not overwrite")
            else:
                with output_path.open("xb") as f:
                    f.write(raw)
            result.update({"extraction_status": "EXTRACTED", "qc_status": qc,
                           "extraction_method": method, "word_count": str(words),
                           "sha256": hash_bytes(raw), "txt_path": str(output_path),
                           "error_message": ""})
        except Exception as exc:
            result["error_message"] = f"{type(exc).__name__}: {exc}"[:600]
        existing[key] = result
        progress = pd.DataFrame(list(existing.values()), columns=columns)
        atomic_csv(progress, checkpoint_path)
        print(f"{r['company_name']} FY{r['fiscal_year']} {form}: {result['extraction_status']} {result['qc_status']}")
    return pd.DataFrame(list(existing.values()), columns=columns)


# ===========================================================
# EXCEL REPORT, TEXT ZIP, HANDOVER AUDIT
# ===========================================================

def safe_excel(value):
    if value is None:
        return ""
    s = str(value)
    if s.lstrip().startswith(("=", "+", "-", "@")):
        return "'" + s
    return s


def label_for_coverage(r):
    if r["coverage_flag"] == "SEC_FORM_FOUND":
        return r["annual_form"]
    aliases = {"CIK_UNRESOLVED": "CIK unresolved", "CIK_AMBIGUOUS": "CIK ambiguous",
               "NO_FORM_IN_MATCHED_CIK": "No form in matched CIK",
               "INVENTORY_ERROR": "Inventory error", "AMBIGUOUS_MULTIPLE_FILINGS": "Multiple forms — review",
               "FY_ASSIGNMENT_REVIEW": "FY assignment — review",
               "PREDECESSOR_CANDIDATE": "Predecessor candidate — review", "PREDECESSOR_AMBIGUOUS": "Predecessor multiple — review"}
    return aliases.get(r["coverage_flag"], r["coverage_flag"])


def split_paragraphs(text):
    # Preserve raw text unchanged in TXT; this derived representation is provisional.
    blocks = [s.strip() for s in re.split(r"\n[ \t]*\n+", text) if s.strip()]
    mode = "BLANK_LINE_BLOCKS" if len(blocks) > 1 else "SINGLE_BLOCK_REVIEW"
    out = []
    for block in blocks:
        if len(block) <= 30000:
            out.append(block)
            continue
        # Excel's 32,767-char cell limit. Break derived blocks without modifying raw.
        remaining = block
        while len(remaining) > 30000:
            ix = remaining.rfind(" ", 10000, 30000)
            ix = 30000 if ix < 10000 else ix
            out.append(remaining[:ix])
            remaining = remaining[ix:].lstrip()
        if remaining:
            out.append(remaining)
        mode += ";SPLIT_CELL_LIMIT"
    return out, mode


def build_exports(coverage, progress, project_root, first_fy, last_fy, known_issues=None):
    known_issues = known_issues or {}
    root = Path(project_root)
    dest = root / "deliverables"
    dest.mkdir(parents=True, exist_ok=True)
    manifest = progress.copy()
    if manifest.empty:
        manifest = pd.DataFrame(columns=["accession_number", "extraction_status", "qc_status", "txt_path", "sha256"])
    def progress_key(record):
        return (str(record["company_id"]), str(record["fiscal_year"]),
                str(record["accession_number"]))
    pmap = {progress_key(r): r for _, r in manifest.iterrows()}
    if len(pmap) != len(manifest):
        raise ValueError("Duplicate company/year/accession in progress during export")
    companies = coverage.drop_duplicates("company_id").sort_values("company_name")
    lookup = {(r["company_id"], r["fiscal_year"]): r for _, r in coverage.iterrows()}
    wb = Workbook()
    yearly = wb.active
    yearly.title = "Company x Year"
    summary = wb.create_sheet("Executive Summary")
    filings_sheet = wb.create_sheet("Filing Inventory")
    paragraphs_sheet = wb.create_sheet("Paragraphs PROVISIONAL")
    qc = wb.create_sheet("QC Exceptions")
    definitions = wb.create_sheet("Definitions")
    years = [str(y) for y in range(first_fy, last_fy + 1)]
    yearly.append(["Company", "Country", "Ticker(s)"] + [f"FY{y}" for y in years]
                   + ["10-K years", "20-F years", "40-F years", "Unresolved / review years"])
    for _, company in companies.iterrows():
        records = [lookup[(company["company_id"], y)] for y in years]
        labels = [label_for_coverage(r) for r in records]
        yearly.append([safe_excel(company["company_name"]), safe_excel(company["country"]),
                       safe_excel(company["ticker"]), *labels,
                       labels.count("10-K"), labels.count("20-F"), labels.count("40-F"),
                       sum(label not in SUPPORTED_FORMS for label in labels)])
    counters = Counter(coverage["coverage_flag"])
    form_counts = Counter(coverage[coverage["coverage_flag"] == "SEC_FORM_FOUND"]["annual_form"])
    extracted = manifest[manifest["extraction_status"] == "EXTRACTED"]
    summary_lines = [
        ["SEC ANNUAL FILINGS | HANDOVER OVERVIEW", ""],
        ["Company universe", len(companies)],
        ["Company-years", len(coverage)],
        ["FY range", f"{first_fy}–{last_fy}"],
        ["SEC 10-K identified", form_counts["10-K"]],
        ["SEC 20-F identified", form_counts["20-F"]],
        ["SEC 40-F identified (not extracted)", form_counts["40-F"]],
        ["10-K texts saved", sum(extracted["annual_form"] == "10-K")],
        ["20-F candidates saved — ALL REQUIRE REVIEW", sum(extracted["annual_form"] == "20-F")],
        ["Extraction errors", sum(manifest["extraction_status"] == "ERROR")],
        ["CIK unresolved", counters["CIK_UNRESOLVED"]],
        ["Matched CIK, no annual form", counters["NO_FORM_IN_MATCHED_CIK"]],
        ["Ambiguous / metadata / inventory failures", sum(v for k,v in counters.items() if k not in ("SEC_FORM_FOUND","CIK_UNRESOLVED","NO_FORM_IN_MATCHED_CIK"))],
        ["IMPORTANT", "A found SEC form is not proof the risk section was fully or correctly extracted."],
        ["20-F NOTE", "20-F Item 3.D parser is a provisional fallback, not validated against prior 20-F dataset."],
        ["UNIVERSE NOTE", "Company list is user-supplied; do not call it the verified official S&P Global 100 roster without independent validation."],
        ["Source", "SEC EDGAR company_tickers.json and per-CIK submissions JSON"],
        ["Generated UTC", now_utc()],
    ]
    for pair in summary_lines:
        summary.append(pair)
    cols = ["company_id", "company_name", "fiscal_year", "coverage_flag", "annual_form",
            "cik", "accession_number", "sec_url", "report_date", "source_of_match",
            "extraction_status", "qc_status", "extraction_method", "word_count", "error_message",
            "processing_decision", "known_issue_note"]
    filings_sheet.append(cols)
    paragraphs_sheet.append(["Company ID", "Company", "Fiscal year", "Form", "Paragraph / block #",
                             "Text", "Segmentation status", "QC status", "Accession", "SEC URL", "TXT path"])
    qc.append(cols)
    for _, r in coverage.iterrows():
        data = pmap.get(progress_key(r), {})
        ext = (lambda x: data.get(x, ""))
        decision, issue = known_issues.get(r["accession_number"], ("INCLUDE_PROVISIONAL", ""))
        row = [r.get(x, "") for x in cols[:10]] + [ext(x) for x in cols[10:15]] + [decision, issue]
        filings_sheet.append([safe_excel(x) for x in row])
        if r["coverage_flag"] != "SEC_FORM_FOUND" or ext("extraction_status") == "ERROR" or (
            ext("extraction_status") == "EXTRACTED" and ext("qc_status") != "PASSED_BASIC_CHECKS") or issue:
            qc.append([safe_excel(x) for x in row])
        if ext("extraction_status") != "EXTRACTED" or decision in ("EXCLUDE_ERROR", "HOLD_FOR_REVIEW"):
            continue
        path = Path(ext("txt_path"))
        raw = path.read_bytes()
        if hash_bytes(raw) != ext("sha256"):
            raise RuntimeError(f"TXT hash mismatch before Excel export: {path}")
        blocks, segmentation = split_paragraphs(raw.decode("utf-8"))
        for i, block in enumerate(blocks, 1):
            paragraphs_sheet.append([safe_excel(r["company_id"]), safe_excel(r["company_name"]),
                                     r["fiscal_year"], r["annual_form"], i, safe_excel(block), segmentation,
                                     ext("qc_status"), r["accession_number"], r["sec_url"], str(path)])
    for pair in [
        ["Field", "Explanation"],
        ["SEC_FORM_FOUND", "Annual filing identified by SEC submissions API under matched CIK and reportDate year"],
        ["NO_FORM_IN_MATCHED_CIK", "No 10-K/20-F/40-F matched this CIK and year; does NOT prove company did not file elsewhere"],
        ["CIK_UNRESOLVED", "Ticker/CIK not found; absence of SEC filings cannot be inferred"],
        ["CIK_AMBIGUOUS", "Ticker maps to multiple CIKs, requires manual confirmation"],
        ["AMBIGUOUS_MULTIPLE_FILINGS", "More than one qualifying filing in company-year, needs manual override"],
        ["FY_ASSIGNMENT_REVIEW", "SEC reportDate may be incorrect or stale; inspect and override"],
        ["PREDECESSOR_CANDIDATE", "Historical CIK filing found; do not assume issuer comparability or extract without manual review"],
        ["KNOWN ISSUES", "Known-bad text is kept in raw TXT and ZIP, excluded from paragraph staging when flagged"],
        ["40-F", "Form inventoried but its risk narrative is not automatically extracted"],
        ["10-K", "EdgarTools TenK.risk_factors, with bounded filing.text fallback for empty sections"],
        ["20-F", "Candidate section using Item 3.D / D. Risk Factors boundaries; every output needs source review"],
        ["Paragraphs PROVISIONAL", "Blank-line-delimited text blocks, NOT guaranteed semantic paragraphs"],
        ["QC", "Warnings indicate manual review required, not proof of an error"],
        ["Source docs", "https://www.sec.gov/search-filings/edgar-application-programming-interfaces"],
        ["API docs", "https://edgartools.readthedocs.io/en/latest/guides/working-with-filing/"],
    ]:
        definitions.append(pair)
    palette = {"10-K": "E4F4EB", "20-F": "E3EEFC", "40-F": "E6E4F9",
               "No form in matched CIK": "FDE8E8", "CIK unresolved": "E9EDF2",
               "CIK ambiguous": "FFF1D4", "Multiple forms — review": "FFF1D4", "FY assignment — review": "FFF1D4"}
    for sheet in wb.worksheets:
        sheet.freeze_panes = "D2" if sheet is yearly else "A2"
        sheet.sheet_view.showGridLines = False
        sheet.row_dimensions[1].height = 30
        for cell in sheet[1]:
            cell.fill = PatternFill("solid", fgColor="19365B")
            cell.font = Font(bold=True, color="FFFFFF")
            cell.alignment = Alignment(vertical="center", wrap_text=True)
        sheet.auto_filter.ref = sheet.dimensions
        for line in sheet.iter_rows(min_row=2):
            for cell in line:
                cell.alignment = Alignment(vertical="top", wrap_text=(sheet is not yearly))
    yearly.column_dimensions["A"].width=36
    yearly.column_dimensions["B"].width=17
    yearly.column_dimensions["C"].width=16
    for column in range(4,4+len(years)):
        yearly.column_dimensions[get_column_letter(column)].width=20
    for line in yearly.iter_rows(min_row=2):
        for cell in line[3:3+len(years)]:
            color = palette.get(cell.value)
            if color:
                cell.fill = PatternFill("solid", fgColor=color)
    summary.column_dimensions["A"].width = 48
    summary.column_dimensions["B"].width = 110
    for sheet in (filings_sheet, qc):
        for column in range(1, sheet.max_column+1):
            sheet.column_dimensions[get_column_letter(column)].width = 25 if column != 2 else 36
    paragraphs_sheet.column_dimensions["A"].width = 22
    paragraphs_sheet.column_dimensions["B"].width = 32
    paragraphs_sheet.column_dimensions["F"].width = 85
    for letter in ("G", "H", "J", "K"):
        paragraphs_sheet.column_dimensions[letter].width = 30
    definitions.column_dimensions["A"].width = 35
    definitions.column_dimensions["B"].width = 115
    filename = dest / "SEC_Annual_Risk_Factors_Overview.xlsx"
    wb.save(filename)
    check = load_workbook(filename, read_only=True)
    try:
        assert check["Company x Year"].max_row == len(companies)+1
    finally:
        check.close()
    return filename


def make_txt_zip(project_root, checkpoint):
    root = Path(project_root)
    outputs = root / "deliverables"
    outputs.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    dest = outputs / f"SEC_Risk_Factors_TXT_Backup_{timestamp}.zip"
    if dest.exists():
        # Rerunning exports within the same second must not overwrite a backup.
        dest = outputs / f"SEC_Risk_Factors_TXT_Backup_{timestamp}_{os.getpid()}.zip"
    inventory = checkpoint.copy()
    rows = []
    with zipfile.ZipFile(dest, "x", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for _, r in inventory.iterrows():
            if r["extraction_status"] != "EXTRACTED":
                continue
            src = Path(r["txt_path"])
            raw = src.read_bytes()
            digest = hash_bytes(raw)
            if digest != r["sha256"]:
                raise RuntimeError(f"Corrupt TXT: {src}")
            arc = str(src.relative_to(root)).replace(os.sep, "/")
            archive.writestr(arc, raw)
            rows.append({"company":r["company_name"], "fiscal_year":r["fiscal_year"],
                         "form":r["annual_form"], "accession":r["accession_number"],
                         "archive_path":arc, "sha256":digest, "qc_status":r["qc_status"]})
        buffer = io.StringIO()
        pd.DataFrame(rows, columns=["company","fiscal_year","form","accession","archive_path","sha256","qc_status"]).to_csv(buffer,index=False)
        archive.writestr("TXT_inventory.csv", buffer.getvalue())
        for name in ("company_universe.csv", "manual_overrides.csv"):
            path = root / "input" / name
            if path.is_file():
                archive.write(path, f"input/{name}")
        for name in ("company_resolution.csv", "filing_inventory.csv", "company_year_coverage.csv", "extraction_progress.csv"):
            path = root / "coverage" / name
            if path.is_file():
                archive.write(path, f"coverage/{name}")
        archive.writestr("README.txt", "All TXT files are original, not overwritten. 20-F texts are unvalidated candidates. Read TXT_inventory.csv for SHA-256 checksums.\n")
    with zipfile.ZipFile(dest) as archive:
        assert archive.testzip() is None
        assert sum(n.startswith("txt/") and n.endswith(".txt") for n in archive.namelist()) == len(rows)
    return dest, len(rows)


## 4. Mount Drive and load YOUR company/ticker file

On the **first run**, upload the *actual* company/ticker CSV or Excel file when prompted. Required columns: `company_name` and `ticker`; optional `country`, `company_id`, `cik`, and `historical_ciks`. The file is copied to `My Drive/SEC_Risk_Handover/input/company_universe.csv` and reused on restart. **No blank CIK column needs to be filled manually.**

The CIK matching step (recovered from original Chunks 11C and 11F) compares both SEC ticker and normalized issuer name. It records ambiguous matches and rejects six known false ticker collisions. The four prior historical CIK linkages (11J/11K) are included as **candidates**; verify each issuer relationship and accession before analytic use. More historical CIKs may remain undiscovered.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import pandas as pd
from IPython.display import display

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
P = project_dirs(ROOT)
company_file = P["input"] / COMPANIES_FILENAME

# IMPORTANT: This code requires YOUR company/ticker list. No fake/sample universe.
needs_upload = not company_file.exists()
if company_file.exists():
    old = pd.read_csv(company_file, dtype=str, keep_default_na=False)
    if len(old) <= 3 and 'company_id' in old.columns and all(
        str(x).startswith('DEMO_') for x in old['company_id']
    ):
        # Migrate the previous handover's demonstration list, not genuine input.
        archived_demo = company_file.with_name('company_universe_PREVIOUS_DEMO.csv')
        if not archived_demo.exists():
            company_file.rename(archived_demo)
        else:
            company_file.unlink()  # Only the known, 3-row DEMO is replaced.
        print('Previous demonstration file archived:', archived_demo.name)
        needs_upload = True
if needs_upload:
    print("Upload your existing S&P Global 100 company list (.csv or .xlsx).")
    print("Required columns: company_name and ticker. Optional: country, company_id, cik, historical_ciks.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV or XLSX input file.")
    uploaded_name, raw = next(iter(uploaded.items()))
    from io import BytesIO
    if uploaded_name.lower().endswith('.csv'):
        original = pd.read_csv(BytesIO(raw), dtype=str, keep_default_na=False, encoding='utf-8-sig')
    elif uploaded_name.lower().endswith('.xlsx'):
        original = pd.read_excel(BytesIO(raw), dtype=str, keep_default_na=False)
        original = original.fillna('')
    else:
        raise ValueError("Only .csv or .xlsx are supported.")
    aliases = {
        'name':'company_name', 'company':'company_name', 'company name':'company_name',
        'holding_name':'company_name', 'holding name':'company_name',
        'ticker symbol':'ticker','symbol':'ticker', 'security_ticker':'ticker',
        'tickers':'ticker','stock ticker':'ticker'
    }
    original = original.rename(columns={
        col: aliases.get(str(col).strip().lower(), str(col).strip())
        for col in original.columns
    })
    missing={'company_name','ticker'}-set(original.columns)
    if missing:
        raise ValueError(f"Your input is missing: {sorted(missing)}. Columns: {list(original.columns)}")
    assert not original.empty
    original.to_csv(company_file, index=False, encoding='utf-8-sig')
    print('Company list saved to Google Drive:', company_file)
else:
    print('Reusing your saved company list:', company_file)

companies = load_companies(company_file)
print('Input company rows:', len(companies))
if len(companies) != 100:
    print('NOTE: Expected approximately 100 unique companies in the historical IOO-derived project.')
    print('Check share-class duplicates and issuer identity; do not silently drop rows.')

# Six verified FY metadata exceptions from the original research notebook (11I).
seed_dir = P['input']
fy_seed = pd.DataFrame([
 ["0000200406-16-000071","0000200406","2016-01-03","2015","JNJ early January FY end"],
 ["0000200406-17-000006","0000200406","2017-01-01","2016","JNJ early January FY end"],
 ["0000200406-21-000008","0000200406","2021-01-03","2020","JNJ early January FY end"],
 ["0000200406-22-000022","0000200406","2022-01-02","2021","JNJ early January FY end"],
 ["0000200406-23-000016","0000200406","2023-01-01","2022","JNJ early January FY end"],
 ["0001373715-18-000058","0001373715","2018-02-28","2017","ServiceNow FY2017 reportDate correction"],
], columns=["accession_number","expected_cik","expected_report_date","corrected_fiscal_year","note"])
known_seed = pd.DataFrame([
 ["0000040545-17-000010","EXCLUDE_ERROR","GE FY2016: annual report beyond Risk Factors"],
 ["0000040545-20-000009","EXCLUDE_ERROR","GE FY2019: begins with Other Financial Data"],
 ["0000104169-17-000021","HOLD_FOR_REVIEW","Walmart FY2017: Item 2 Properties spillover"],
], columns=["accession_number","processing_decision","reason"])
if not (seed_dir/'fiscal_year_corrections.csv').exists():
    atomic_csv(fy_seed, seed_dir/'fiscal_year_corrections.csv')
if not (seed_dir/'known_issues.csv').exists():
    atomic_csv(known_seed, seed_dir/'known_issues.csv')
_ = load_overrides(seed_dir/'manual_overrides.csv')
print('No original files or extraction checkpoints were overwritten.')


## 5. Resolve current CIKs, audit historical CIKs, and discover annual filing coverage

The lookup fetches official SEC `company_tickers.json`, compares company names as well as tickers, records suggested CIKs and ambiguity statuses, retrieves annual submissions history, and tests four separately identified predecessor CIKs. All mapping decisions are auditable CSV files:

- `coverage/company_resolution.csv`: company → current CIK(s), historical candidate CIK(s), directory names, and match status.
- `coverage/cik_review_queue.csv`: ticker/name mismatches, ambiguous CIKs, and manual-review candidates.
- `coverage/historical_cik_filing_candidates.csv`: predecessor filings by fiscal year, accession, and relationship evidence (never automatically certified).
- `coverage/filing_inventory.csv` and `coverage/company_year_coverage.csv`: actual 10-K/20-F filing records and annual coverage.

Historical CIKs **cannot be automatically inferred exhaustively** from a ticker: SEC filer CIKs persist through name changes, but mergers/redomiciliations may create distinct registrants. Cases not in the prior historical investigation remain an explicit research gap. `RUN_DISCOVERY=False` on restart reuses your saved CSVs.


In [ ]:
import pandas as pd
from IPython.display import display
from importlib.metadata import version, PackageNotFoundError

companies = load_companies(P["input"] / COMPANIES_FILENAME)
if MODE == "full" and any(companies["company_id"].str.startswith("DEMO_")):
    raise ValueError("Replace DEMO company_universe.csv before MODE='full'.")

if RUN_DISCOVERY:
    client = SecClient(P["cache"], sec_identity(), refresh=REFRESH_SEC_METADATA)
    ticker_json = client.json(
        "https://www.sec.gov/files/company_tickers.json", "company_tickers.json"
    )
    resolved = resolve_companies(companies, ticker_json)
    atomic_csv(resolved, P["coverage"] / "company_resolution.csv")
    atomic_csv(resolved.loc[resolved["resolution_status"].str.contains(
        "REVIEW|AMBIGUOUS|UNRESOLVED|REJECTED", regex=True, na=False)],
        P["coverage"] / "cik_review_queue.csv")
    print('Historical candidate issuers:',
          int(resolved['historical_status'].eq('PREDECESSOR_CANDIDATE_REVIEW').sum()))
    print("CIK resolution:\n", resolved["resolution_status"].value_counts())

    filing_inventory, lookup_errors = discover_inventory(
        resolved, client, START_FY, END_FY
    )
    filing_inventory = apply_fiscal_year_corrections(
        filing_inventory, P["input"] / "fiscal_year_corrections.csv"
    )
    atomic_csv(filing_inventory, P["coverage"] / "filing_inventory.csv")
    historical_audit = historical_cik_audit(resolved, filing_inventory, START_FY, END_FY)
    atomic_csv(historical_audit, P["coverage"] / "historical_cik_filing_candidates.csv")
    print('Historical predecessor candidate filings:',
          int(historical_audit['status'].eq('PREDECESSOR_CANDIDATE_REVIEW').sum()))
    atomic_csv(lookup_errors, P["coverage"] / "inventory_lookup_errors.csv")
    overrides = load_overrides(P["input"] / "manual_overrides.csv")
    coverage = compile_coverage(
        resolved, filing_inventory, lookup_errors,
        overrides, START_FY, END_FY
    )
    atomic_csv(coverage, P["coverage"] / "company_year_coverage.csv")
else:
    coverage_path = P["coverage"] / "company_year_coverage.csv"
    if not coverage_path.is_file():
        raise FileNotFoundError("No saved coverage CSV: set RUN_DISCOVERY=True first.")
    coverage = pd.read_csv(coverage_path, dtype=str, keep_default_na=False)
    resolved = pd.read_csv(
        P["coverage"] / "company_resolution.csv",
        dtype=str, keep_default_na=False
    )
    print("Restored cached project coverage. No SEC metadata requests.")

print("\nCompany-years:", len(coverage))
print("Coverage flags:\n", coverage["coverage_flag"].value_counts())
print("Annual form counts:\n", coverage["annual_form"].value_counts())
for pkg in ("edgartools", "pandas", "openpyxl", "requests"):
    try:
        print("Package", pkg, version(pkg))
    except PackageNotFoundError:
        pass


## 6. Run 10-K and 20-F extraction with continuous checkpointing

Accession-locked verification: match **CIK and form** before extraction. 10-K uses the proven EdgarTools `filing.obj().risk_factors` path, falling back to bounded whole-filing text / HTML (McDonald's FY2020–2022 pattern). **20-F is only a candidate** from `risk_factors` when available or a bounded Item 3.D → Item 4 section; all 20-F archives are marked `REVIEW_20F_UNVALIDATED`. A failed boundary match is an ERROR, not a successful empty section.

`txt/` is write-once. The CSV checkpoint is saved after each attempted filing; restart rechecks SHA-256 and skips completed records. The notebook performs a **small pilot** until you change MODE to `full` in Step 2.


In [ ]:
progress = process_extractions(
    coverage,
    P["txt"],
    P["coverage"] / "extraction_progress.csv",
    mode=MODE,
    per_form=PILOT_PER_FORM,
    retry_errors=RETRY_ERRORS,
)
print("\nExtraction status:\n", progress["extraction_status"].value_counts())
print("QC status:\n", progress["qc_status"].value_counts())
print("Errors:\n", progress.loc[
    progress["extraction_status"].eq("ERROR"),
    ["company_name", "fiscal_year", "annual_form", "error_message"]
].head(20))


## 7. Excel overview + provisional text blocks + verified original TXT ZIP

The workbook has six sheets: `Company x Year`, `Executive Summary`, `Filing Inventory`, `Paragraphs PROVISIONAL`, `QC Exceptions`, `Definitions`. Exported text blocks are **not guaranteed linguistic paragraphs** because many filing processors remove paragraph boundaries. A suspected error should be logged in `input/known_issues.csv` to exclude or hold that accession in the derived paragraph tab. The raw TXT stays untouched and appears in the ZIP even when flagged.

The output ZIP is organised by `form/company_id/FY/accession` and includes an SHA-256 manifest; both Excel and ZIP are saved on the colleague's own Drive. Download only if you want a local copy.


In [ ]:
known_issues = read_known_issues(P["input"] / "known_issues.csv")
xlsx_path = build_exports(
    coverage, progress, ROOT, START_FY, END_FY,
    known_issues=known_issues
)
zip_path, text_count = make_txt_zip(ROOT, progress)
print("\nExcel workbook:", xlsx_path)
print("Original TXT ZIP:", zip_path)
print("Verified TXT files in ZIP:", text_count)
print("Optional manual browser downloads:")
print("  from google.colab import files")
print("  files.download(str(xlsx_path))")
print("  files.download(str(zip_path))")


## 8. Quality-control handoff (required)

1. Compare **start and end** of sampled TXT sections against the SEC source URLs in `coverage/company_year_coverage.csv` — not only TXT hash integrity.
2. Review documents with `REVIEW_START`, `REVIEW_END`, unusually long text, missing paragraphs and two-column reading-order issues; do not assume `PASSED_BASIC_CHECKS` means fully accurate.
3. Check every **20-F** extraction and mark questionable accessions `HOLD_FOR_REVIEW` or `EXCLUDE_ERROR` in `input/known_issues.csv`; these candidates have *not* been validated historically.
4. Known historical cases from the original notebook: **GE FY2016** excess annual-report text; **GE FY2019** preceding financial data; **Walmart FY2017** Item 2 Properties spillover. They are automatically held/excluded from the provisional paragraph workbook when matching accessions appear.


In [ ]:
print("Discovered annual forms:\n", coverage["annual_form"].value_counts())
print("Saved extraction records:\n", progress.loc[
    progress["extraction_status"].eq("EXTRACTED"), "annual_form"
].value_counts())
check_cols = ["company_name", "fiscal_year", "annual_form", "qc_status", "word_count", "sec_url"]
flagged = progress[
    progress["extraction_status"].eq("ERROR") |
    (~progress["qc_status"].eq("PASSED_BASIC_CHECKS"))
]
print("Cases for manual review:", len(flagged))
display(flagged[check_cols].head(30))


## 9. Restart without running extraction again

Run **Install → Configuration → Reusable functions → Mount Drive** to recreate Python functions and paths. Google Drive retains company input, CIK matches, SEC submissions cache, TXT and checkpoint. Set `RUN_DISCOVERY=False` to reuse existing coverage; then continue extraction in full mode (completed accessions hash-checked and skipped) or run the separate export-only cell. Do not reupload a company list unless it changed. If changed, set `RUN_DISCOVERY=True` and verify existing CIK/coverage mapping before resuming.


In [ ]:
# OPTIONAL RESTART: load saved checkpoint/coverage; no SEC requests.
# Run after Steps 1–4, with RUN_DISCOVERY=False, if you only want exports.
coverage = pd.read_csv(
    P["coverage"] / "company_year_coverage.csv",
    dtype=str, keep_default_na=False
)
checkpoint_file = P["coverage"] / "extraction_progress.csv"
progress = pd.read_csv(checkpoint_file, dtype=str, keep_default_na=False) if checkpoint_file.exists() else pd.DataFrame()
print("Restored rows:", len(coverage), "coverage;", len(progress), "extraction progress")
